# Cross-Validation Comparison

5-fold CV with per-user split within each fold — guarantees every user appears in both train and test across all folds.

**Models compared:**

| Model | Type |
|---|---|
| User+Item Bias | Non-ML baseline, no latent factors |
| CF (SGD) | Matrix Factorization via gradient descent |
| CF (SVD) | Matrix Factorization via closed-form decomposition |
| Content-Based | Item-item cosine similarity on genre + year |

**Metrics:** RMSE, MAE, Precision@10, Recall@10

**Why CV here?** Single train/test split results depend on which ratings happened to land in the test set. CV averages across 5 different splits, giving statistically reliable estimates and confidence intervals on the improvement of each model over the baseline.


## 1. Imports & Load Data

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import json

BG, PANEL, GRID_C = "#0f0f14", "#16161f", "#2a2a35"
TEXT_C, ACCENT, ACCENT2, ACCENT3, ACCENT4 = "#d4d4e0", "#7DF9C4", "#F97D7D", "#7DA8F9", "#F9E07D"

def style_ax(ax, title):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=TEXT_C, fontsize=10, pad=9, fontweight="bold")
    ax.tick_params(colors=TEXT_C, labelsize=8)
    ax.xaxis.label.set_color(TEXT_C)
    ax.yaxis.label.set_color(TEXT_C)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_C)
    ax.grid(color=GRID_C, linewidth=0.5, linestyle="--", alpha=0.7)

# load full filtered ratings (train + test combined — CV will re-split)
train = pd.read_csv("../data/processed/train.csv")
test  = pd.read_csv("../data/processed/test.csv")
ratings = pd.concat([train, test]).reset_index(drop=True)
movies  = pd.read_csv("../data/processed/movies_clean.csv")

N_USERS  = ratings.user_idx.nunique()
N_MOVIES = ratings.movie_idx.nunique()
RATING_MIN, RATING_MAX = 0.5, 5.0
K = 10

print(f"Total ratings : {len(ratings):,}")
print(f"Users         : {N_USERS}")
print(f"Movies        : {N_MOVIES}")


Total ratings : 81,116
Users         : 610
Movies        : 2269


## 2. Per-User K-Fold Split

For each fold:
- Split each user's ratings into train (80%) and test (20%) independently
- This guarantees every user appears in both splits in every fold
- Different from a global random split which could leave some users cold-start


In [11]:
def temporal_train_test_split(ratings, test_frac=0.2):
    """
    Temporal per-user split.
    Each user's ratings sorted by timestamp — most recent test_frac → test,
    older (1-test_frac) → train. No randomness, no leakage.
    """
    train_rows, test_rows = [], []

    for _, group in ratings.groupby("user_idx"):
        group = group.sort_values("timestamp")
        split = max(1, int(len(group) * test_frac))
        test_rows.extend(group.index[-split:])    # most recent 20%
        train_rows.extend(group.index[:-split])   # older 80%

    fold_train = ratings.loc[train_rows].reset_index(drop=True)
    fold_test  = ratings.loc[test_rows].reset_index(drop=True)
    return fold_train, fold_test


# create single temporal split
temporal_train, temporal_test = temporal_train_test_split(ratings, test_frac=0.2)

print(f"Train : {len(temporal_train):,}")
print(f"Test  : {len(temporal_test):,}")
print(f"Split : {len(temporal_test)/(len(temporal_train)+len(temporal_test)):.1%} test")
print()

# sanity checks
cold = set(temporal_test.user_idx) - set(temporal_train.user_idx)
print(f"✓ Cold-start users in test : {len(cold)}")

train_max_ts = temporal_train.groupby("user_idx")["timestamp"].max()
test_min_ts  = temporal_test.groupby("user_idx")["timestamp"].min()
common       = train_max_ts.index.intersection(test_min_ts.index)
violations   = (train_max_ts[common] > test_min_ts[common]).sum()
print(f"✓ Temporal leakage violations : {violations}")

train_rpu = temporal_train.groupby("user_idx")["rating"].count()
test_rpu  = temporal_test.groupby("user_idx")["rating"].count()
print(f"\nTrain ratings/user — min={train_rpu.min()}  median={train_rpu.median():.0f}  max={train_rpu.max()}")
print(f"Test  ratings/user — min={test_rpu.min()}   median={test_rpu.median():.0f}  max={test_rpu.max()}")

Train : 65,130
Test  : 15,986
Split : 19.7% test

✓ Cold-start users in test : 0
✓ Temporal leakage violations : 0

Train ratings/user — min=6  median=52  max=1308
Test  ratings/user — min=1   median=12  max=326


## 3. Evaluation Metrics

In [12]:
def rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))

def mae(actual, predicted):
    return np.mean(np.abs(actual - predicted))

def precision_recall_at_k(recommend_fn, test_df, train_df, k=10, threshold=4.0):
    train_seen = train_df.groupby("user_idx")["movie_idx"].apply(set).to_dict()
    test_relevant = (
        test_df[test_df["rating"] >= threshold]
        .groupby("user_idx")["movie_idx"]
        .apply(set)
        .to_dict()
    )
    precisions, recalls = [], []
    for user_idx, relevant in test_relevant.items():
        seen    = train_seen.get(user_idx, set())
        recs    = recommend_fn(user_idx, k, seen)
        rec_set = {m for m, _ in recs}
        hits    = len(rec_set & relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant) if relevant else 0.0)
    p = np.mean(precisions)
    r = np.mean(recalls)
    return p, r


## 4. Model Definitions

### 4a. User+Item Bias

In [13]:
class UserItemBias:
    def __init__(self, beta=10):
        self.beta = beta
        self.mu = 0.0
        self.bu = {}
        self.bi = {}

    def fit(self, train_df):
        self.mu = train_df["rating"].mean()

        user_stats = train_df.groupby("user_idx")["rating"].agg(["sum","count"])
        self.bu = ((user_stats["sum"] - user_stats["count"] * self.mu)
                   / (user_stats["count"] + self.beta)).to_dict()

        item_stats = train_df.groupby("movie_idx")["rating"].agg(["sum","count"])
        self.bi = ((item_stats["sum"] - item_stats["count"] * self.mu)
                   / (item_stats["count"] + self.beta)).to_dict()

    def predict_batch(self, user_idxs, movie_idxs):
        bu = np.array([self.bu.get(u, 0.0) for u in user_idxs])
        bi = np.array([self.bi.get(i, 0.0) for i in movie_idxs])
        return self.mu + bu + bi

    def recommend(self, user_idx, n=10, seen_items=None):
        all_movies = np.arange(N_MOVIES)
        if seen_items is not None:
            mask = ~np.isin(all_movies, list(seen_items))
            candidates = all_movies[mask]
        else:
            candidates = all_movies
        scores = np.clip(
            self.mu + self.bu.get(user_idx, 0.0)
            + np.array([self.bi.get(i, 0.0) for i in candidates]),
            RATING_MIN, RATING_MAX
        )
        top_idx = np.argpartition(scores, -n)[-n:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        return list(zip(candidates[top_idx], scores[top_idx]))


### 4b. CF (SGD)

In [14]:
class SGDMatrixFactorization:
    def __init__(self, n_users, n_movies, n_factors=35, lr=0.02, reg=0.12, seed=42):
        self.n_factors = n_factors
        self.lr, self.reg = lr, reg
        rng   = np.random.default_rng(seed)
        scale = 1.0 / np.sqrt(n_factors)
        self.P  = rng.normal(0, scale, (n_users,  n_factors))
        self.Q  = rng.normal(0, scale, (n_movies, n_factors))
        self.bu = np.zeros(n_users)
        self.bi = np.zeros(n_movies)
        self.mu = 0.0

    def predict_one(self, u, i):
        return self.mu + self.bu[u] + self.bi[i] + self.P[u] @ self.Q[i]

    def predict_batch(self, user_idxs, movie_idxs):
        dots = np.einsum("nd,nd->n", self.P[user_idxs], self.Q[movie_idxs])
        return self.mu + self.bu[user_idxs] + self.bi[movie_idxs] + dots

    def fit(self, train_df, n_epochs=38, patience=4):
        self.mu = train_df["rating"].mean()
        users = train_df["user_idx"].values
        items = train_df["movie_idx"].values
        rvals = train_df["rating"].values.astype(float)
        n = len(rvals)
        rng = np.random.default_rng(0)
        best_val, no_improve = np.inf, 0

        for epoch in range(1, n_epochs + 1):
            perm = rng.permutation(n)
            u_s, i_s, r_s = users[perm], items[perm], rvals[perm]
            sq_err = 0.0
            for k in range(n):
                u, i, r = u_s[k], i_s[k], r_s[k]
                err = r - self.predict_one(u, i)
                sq_err += err ** 2
                pu = self.P[u].copy()
                qi = self.Q[i].copy()
                self.bu[u] += self.lr * (err - self.reg * self.bu[u])
                self.bi[i] += self.lr * (err - self.reg * self.bi[i])
                self.P[u]  += self.lr * (err * qi - self.reg * pu)
                self.Q[i]  += self.lr * (err * pu - self.reg * qi)

            train_rmse = np.sqrt(sq_err / n)
            if train_rmse < best_val:
                best_val   = train_rmse
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    break

    def recommend(self, user_idx, n=10, seen_items=None):
        all_movies = np.arange(self.Q.shape[0])
        if seen_items is not None:
            mask = ~np.isin(all_movies, list(seen_items))
            candidates = all_movies[mask]
        else:
            candidates = all_movies
        scores = np.clip(
            self.mu + self.bu[user_idx] + self.bi[candidates]
            + self.P[user_idx] @ self.Q[candidates].T,
            RATING_MIN, RATING_MAX
        )
        top_idx = np.argpartition(scores, -n)[-n:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        return list(zip(candidates[top_idx], scores[top_idx]))


### 4c. CF (SVD)

In [15]:
class SVDModel:
    def __init__(self, k=35, beta=10):
        self.k = k
        self.beta = beta
        self.mu = 0.0
        self.bu = None
        self.bi = None
        self.P  = None
        self.Q  = None

    def fit(self, train_df):
        self.mu = train_df["rating"].mean()
        n_users  = train_df["user_idx"].nunique()
        n_movies = train_df["movie_idx"].nunique()

        # damped biases
        user_stats = train_df.groupby("user_idx")["rating"].agg(["sum","count"])
        user_bias  = ((user_stats["sum"] - user_stats["count"] * self.mu)
                      / (user_stats["count"] + self.beta))
        user_bias  = user_bias.reindex(range(n_users), fill_value=0.0)

        item_stats = train_df.groupby("movie_idx")["rating"].agg(["sum","count"])
        item_bias  = ((item_stats["sum"] - item_stats["count"] * self.mu)
                      / (item_stats["count"] + self.beta))
        item_bias  = item_bias.reindex(range(n_movies), fill_value=0.0)

        self.bu = user_bias.values
        self.bi = item_bias.values

        # residual matrix
        R_resid = np.zeros((n_users, n_movies))
        baseline = self.mu + self.bu[:, None] + self.bi[None, :]
        u_idx = train_df["user_idx"].values
        i_idx = train_df["movie_idx"].values
        r_val = train_df["rating"].values
        R_resid[u_idx, i_idx] = r_val - baseline[u_idx, i_idx]

        # SVD
        U, S, Vt = np.linalg.svd(R_resid, full_matrices=False)
        k = min(self.k, len(S))
        sqrt_S = np.sqrt(S[:k])
        self.P = U[:, :k] * sqrt_S[None, :]
        self.Q = Vt[:k, :].T * sqrt_S[None, :]

    def predict_batch(self, user_idxs, movie_idxs):
        dots = np.einsum("nd,nd->n", self.P[user_idxs], self.Q[movie_idxs])
        return self.mu + self.bu[user_idxs] + self.bi[movie_idxs] + dots

    def recommend(self, user_idx, n=10, seen_items=None):
        all_movies = np.arange(self.Q.shape[0])
        if seen_items is not None:
            mask = ~np.isin(all_movies, list(seen_items))
            candidates = all_movies[mask]
        else:
            candidates = all_movies
        scores = np.clip(
            self.mu + self.bu[user_idx] + self.bi[candidates]
            + self.P[user_idx] @ self.Q[candidates].T,
            RATING_MIN, RATING_MAX
        )
        top_idx = np.argpartition(scores, -n)[-n:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        return list(zip(candidates[top_idx], scores[top_idx]))


### 4d. Content-Based

In [16]:
# Build content matrix once — same across all folds (features don't change)
ALL_GENRES = [
    "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir",
    "Horror", "Musical", "Mystery", "Romance", "Sci-Fi",
    "Thriller", "War", "Western", "IMAX"
]

ALPHA_CB = 0.5   # best from content-based sweep
N_CB     = 30    # best neighbourhood size

genre_matrix = np.zeros((N_MOVIES, len(ALL_GENRES)))
for _, row in movies.iterrows():
    idx    = int(row["movie_idx"])
    genres = str(row["genres"]).split("|")
    for g in genres:
        if g.strip() in ALL_GENRES:
            genre_matrix[idx, ALL_GENRES.index(g.strip())] = 1.0

year_vals = movies.set_index("movie_idx")["year"].reindex(range(N_MOVIES))
min_year, max_year = year_vals.min(), year_vals.max()
norm_year = ((year_vals - min_year) / (max_year - min_year)).fillna(0.5).values

content_matrix = np.hstack([genre_matrix, (ALPHA_CB * norm_year).reshape(-1, 1)])
norms_cm = np.linalg.norm(content_matrix, axis=1, keepdims=True)
norms_cm = np.where(norms_cm == 0, 1e-9, norms_cm)
content_normed = content_matrix / norms_cm
sim_matrix = content_normed @ content_normed.T

print(f"Content matrix : {content_matrix.shape}")
print(f"Sim matrix     : {sim_matrix.shape}")

class ContentBased:
    def __init__(self, sim_matrix, n_neighbours=N_CB):
        self.sim_matrix   = sim_matrix
        self.n_neighbours = n_neighbours
        self.user_ratings = {}
        self.user_means   = {}
        self.global_mean  = 0.0

    def fit(self, train_df):
        self.global_mean  = train_df["rating"].mean()
        self.user_ratings = (
            train_df.groupby("user_idx")
                    .apply(lambda df: dict(zip(df["movie_idx"], df["rating"])))
                    .to_dict()
        )
        self.user_means = train_df.groupby("user_idx")["rating"].mean().to_dict()

    def predict_one(self, user_idx, movie_idx):
        rated = self.user_ratings.get(user_idx, {})
        if not rated:
            return self.user_means.get(user_idx, self.global_mean)
        rated_idxs   = np.array(list(rated.keys()))
        rated_ratings = np.array(list(rated.values()), dtype=float)
        sims         = self.sim_matrix[movie_idx, rated_idxs]
        top_mask     = np.argsort(sims)[-self.n_neighbours:]
        top_sims     = sims[top_mask]
        top_ratings  = rated_ratings[top_mask]
        pos          = top_sims > 0
        if pos.sum() == 0:
            return self.user_means.get(user_idx, self.global_mean)
        return np.dot(top_sims[pos], top_ratings[pos]) / top_sims[pos].sum()

    def predict_batch(self, user_idxs, movie_idxs):
        return np.array([self.predict_one(u, i) for u, i in zip(user_idxs, movie_idxs)])

    def recommend(self, user_idx, n=10, seen_items=None):
        all_movies = np.arange(N_MOVIES)
        if seen_items is not None:
            mask = ~np.isin(all_movies, list(seen_items))
            candidates = all_movies[mask]
        else:
            candidates = all_movies
        scores = np.clip(
            np.array([self.predict_one(user_idx, i) for i in candidates]),
            RATING_MIN, RATING_MAX
        )
        top_idx = np.argpartition(scores, -n)[-n:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        return list(zip(candidates[top_idx], scores[top_idx]))


Content matrix : (2269, 20)
Sim matrix     : (2269, 2269)


In [17]:
def run_temporal_comparison(temporal_train, temporal_test, k=K):
    model_names = ["User+Item Bias", "CF (SGD)", "CF (SVD)", "Content-Based"]
    results = {name: {"rmse": None, "mae": None, "precision": None, "recall": None}
               for name in model_names}

    actual = temporal_test["rating"].values

    print(f"{'='*55}")
    print(f"  Temporal Split Evaluation")
    print(f"  train={len(temporal_train):,}  test={len(temporal_test):,}")
    print(f"{'='*55}")

    # ── User+Item Bias ─────────────────────────────────────
    t0   = time.time()
    bias = UserItemBias()
    bias.fit(temporal_train)
    preds = np.clip(bias.predict_batch(temporal_test["user_idx"].values,
                                       temporal_test["movie_idx"].values),
                    RATING_MIN, RATING_MAX)
    p, r = precision_recall_at_k(bias.recommend, temporal_test, temporal_train, k=k)
    results["User+Item Bias"] = {"rmse": rmse(actual, preds), "mae": mae(actual, preds),
                                  "precision": p, "recall": r}
    print(f"  Bias      RMSE={results['User+Item Bias']['rmse']:.4f}  "
          f"MAE={results['User+Item Bias']['mae']:.4f}  "
          f"P@{k}={p:.4f}  R@{k}={r:.4f}  ({time.time()-t0:.1f}s)")

    # ── CF SGD ─────────────────────────────────────────────
    t0  = time.time()
    sgd = SGDMatrixFactorization(N_USERS, N_MOVIES)
    sgd.fit(temporal_train)
    preds = np.clip(sgd.predict_batch(temporal_test["user_idx"].values,
                                      temporal_test["movie_idx"].values),
                    RATING_MIN, RATING_MAX)
    p, r  = precision_recall_at_k(sgd.recommend, temporal_test, temporal_train, k=k)
    results["CF (SGD)"] = {"rmse": rmse(actual, preds), "mae": mae(actual, preds),
                            "precision": p, "recall": r}
    print(f"  CF SGD    RMSE={results['CF (SGD)']['rmse']:.4f}  "
          f"MAE={results['CF (SGD)']['mae']:.4f}  "
          f"P@{k}={p:.4f}  R@{k}={r:.4f}  ({time.time()-t0:.1f}s)")

    # ── CF SVD ─────────────────────────────────────────────
    t0  = time.time()
    svd = SVDModel(k=35)
    svd.fit(temporal_train)
    preds = np.clip(svd.predict_batch(temporal_test["user_idx"].values,
                                      temporal_test["movie_idx"].values),
                    RATING_MIN, RATING_MAX)
    p, r  = precision_recall_at_k(svd.recommend, temporal_test, temporal_train, k=k)
    results["CF (SVD)"] = {"rmse": rmse(actual, preds), "mae": mae(actual, preds),
                            "precision": p, "recall": r}
    print(f"  CF SVD    RMSE={results['CF (SVD)']['rmse']:.4f}  "
          f"MAE={results['CF (SVD)']['mae']:.4f}  "
          f"P@{k}={p:.4f}  R@{k}={r:.4f}  ({time.time()-t0:.1f}s)")

    # ── Content-Based ──────────────────────────────────────
    t0 = time.time()
    cb = ContentBased(sim_matrix)
    cb.fit(temporal_train)
    preds = np.clip(cb.predict_batch(temporal_test["user_idx"].values,
                                     temporal_test["movie_idx"].values),
                    RATING_MIN, RATING_MAX)
    p, r  = precision_recall_at_k(cb.recommend, temporal_test, temporal_train, k=k)
    results["Content-Based"] = {"rmse": rmse(actual, preds), "mae": mae(actual, preds),
                                  "precision": p, "recall": r}
    print(f"  Content   RMSE={results['Content-Based']['rmse']:.4f}  "
          f"MAE={results['Content-Based']['mae']:.4f}  "
          f"P@{k}={p:.4f}  R@{k}={r:.4f}  ({time.time()-t0:.1f}s)")
      
    # ── Random Baseline ────────────────────────────────────
    t0  = time.time()
    rng = np.random.default_rng(42)

    # RMSE/MAE — uniform random predictions
    random_preds = rng.uniform(RATING_MIN, RATING_MAX, size=len(actual))
    r_rmse = rmse(actual, random_preds)
    r_mae  = mae(actual, random_preds)

    # P@K / R@K — sample K random unseen movies per user
    train_seen = temporal_train.groupby("user_idx")["movie_idx"].apply(set).to_dict()
    test_relevant = (
        temporal_test[temporal_test["rating"] >= 4.0]
        .groupby("user_idx")["movie_idx"]
        .apply(set)
        .to_dict()
    )
    precisions, recalls = [], []
    for user_idx, relevant in test_relevant.items():
        seen    = train_seen.get(user_idx, set())
        unseen  = np.array(list(set(range(N_MOVIES)) - seen))
        k_sample = min(k, len(unseen))
        recs    = set(rng.choice(unseen, size=k_sample, replace=False))
        hits    = len(recs & relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant) if relevant else 0.0)

    p, r = np.mean(precisions), np.mean(recalls)
    results["Random"] = {"rmse": r_rmse, "mae": r_mae,
                         "precision": p, "recall": r}
    print(f"  Random    RMSE={r_rmse:.4f}  MAE={r_mae:.4f}  "
          f"P@{k}={p:.4f}  R@{k}={r:.4f}  ({time.time()-t0:.1f}s)")

    return results

temporal_results = run_temporal_comparison(temporal_train, temporal_test)

  Temporal Split Evaluation
  train=65,130  test=15,986
  Bias      RMSE=0.8760  MAE=0.6703  P@10=0.0446  R@10=0.0386  (0.2s)
  CF SGD    RMSE=0.8500  MAE=0.6494  P@10=0.0136  R@10=0.0108  (19.0s)
  CF SVD    RMSE=0.8693  MAE=0.6641  P@10=0.0544  R@10=0.0447  (6.2s)
  Content   RMSE=0.9160  MAE=0.7040  P@10=0.0121  R@10=0.0071  (21.6s)
  Random    RMSE=1.8347  MAE=1.5007  P@10=0.0071  R@10=0.0037  (0.1s)


## 6. Results Summary

In [19]:
def summarise(cv_results):
    """Compute mean ± std across folds for each model and metric."""
    rows = []
    for model_name, metrics in cv_results.items():
        row = {"Model": model_name}
        for metric, vals in metrics.items():
            row[f"{metric}_mean"] = np.mean(vals)
            row[f"{metric}_std"]  = np.std(vals)
        rows.append(row)
    return pd.DataFrame(rows)

summary = summarise(temporal_results)

print("=" * 75)
print(f"  {'Model':<20} {'RMSE':>12} {'MAE':>12} {'P@10':>12} {'R@10':>12}")
print("=" * 75)
for _, row in summary.iterrows():
    if row["Model"] == "Random":
        # only show ranking metrics for random — RMSE/MAE not meaningful
        print(f"  {row['Model']:<20} {'—':>12} {'—':>12} "
              f"{row['precision_mean']:.4f}±{row['precision_std']:.4f}  "
              f"{row['recall_mean']:.4f}±{row['recall_std']:.4f}")
    else:
        print(f"  {row['Model']:<20} "
              f"{row['rmse_mean']:.4f}±{row['rmse_std']:.4f}  "
              f"{row['mae_mean']:.4f}±{row['mae_std']:.4f}  "
              f"{row['precision_mean']:.4f}±{row['precision_std']:.4f}  "
              f"{row['recall_mean']:.4f}±{row['recall_std']:.4f}")
print("=" * 75)

# RMSE improvement over User+Item Bias
random_rmse    = summary[summary.Model == "Random"]["rmse_mean"].values[0]
baseline_rmse  = summary[summary.Model == "User+Item Bias"]["rmse_mean"].values[0]
random_p       = summary[summary.Model == "Random"]["precision_mean"].values[0]

print("\nRMSE improvement over User+Item Bias:")
for _, row in summary.iterrows():
    if row["Model"] == "User+Item Bias":
        continue
    delta = baseline_rmse - row["rmse_mean"]
    pct   = delta / baseline_rmse * 100
    print(f"  {row['Model']:<20} Δ={delta:+.4f}  ({pct:+.2f}%)")


print("\nP@10 improvement over Random:")
for _, row in summary.iterrows():
    if row["Model"] == "Random":
        continue
    delta = row["precision_mean"] - random_p
    mult  = row["precision_mean"] / random_p if random_p > 0 else float("inf")
    print(f"  {row['Model']:<20} Δ={delta:+.4f}  ({mult:.1f}× random)")

  Model                        RMSE          MAE         P@10         R@10
  User+Item Bias       0.8760±0.0000  0.6703±0.0000  0.0446±0.0000  0.0386±0.0000
  CF (SGD)             0.8500±0.0000  0.6494±0.0000  0.0136±0.0000  0.0108±0.0000
  CF (SVD)             0.8693±0.0000  0.6641±0.0000  0.0544±0.0000  0.0447±0.0000
  Content-Based        0.9160±0.0000  0.7040±0.0000  0.0121±0.0000  0.0071±0.0000
  Random                          —            — 0.0071±0.0000  0.0037±0.0000

RMSE improvement over User+Item Bias:
  CF (SGD)             Δ=+0.0260  (+2.97%)
  CF (SVD)             Δ=+0.0067  (+0.76%)
  Content-Based        Δ=-0.0400  (-4.57%)
  Random               Δ=-0.9588  (-109.45%)

P@10 improvement over Random:
  User+Item Bias       Δ=+0.0374  (6.2× random)
  CF (SGD)             Δ=+0.0065  (1.9× random)
  CF (SVD)             Δ=+0.0473  (7.6× random)
  Content-Based        Δ=+0.0049  (1.7× random)


## 9. Save Results

In [21]:
with open("temporal_results.json", "w") as f:
    json.dump(
        {model: {metric: float(val) for metric, val in metrics.items()}
         for model, metrics in temporal_results.items()},
        f, indent=2
    )

print("Saved: temporal_results.json")
print()
print(json.dumps(
    {model: {metric: round(float(val), 4) for metric, val in metrics.items()}
     for model, metrics in temporal_results.items()},
    indent=2
))

Saved: temporal_results.json

{
  "User+Item Bias": {
    "rmse": 0.876,
    "mae": 0.6703,
    "precision": 0.0446,
    "recall": 0.0386
  },
  "CF (SGD)": {
    "rmse": 0.85,
    "mae": 0.6494,
    "precision": 0.0136,
    "recall": 0.0108
  },
  "CF (SVD)": {
    "rmse": 0.8693,
    "mae": 0.6641,
    "precision": 0.0544,
    "recall": 0.0447
  },
  "Content-Based": {
    "rmse": 0.916,
    "mae": 0.704,
    "precision": 0.0121,
    "recall": 0.0071
  },
  "Random": {
    "rmse": 1.8347,
    "mae": 1.5007,
    "precision": 0.0071,
    "recall": 0.0037
  }
}
